In [1]:
import os
import glob
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma
from ultralytics import YOLO
import ollama

print("Libraries loaded successfully")

C:\Users\clo19\AppData\Local\Temp\ipykernel_26568\1872348656.py:3: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


Libraries loaded successfully


In [2]:
doc_paths = glob.glob("../data/docs/*.pdf")
documents = []

for path in doc_paths:
    loader = PyPDFLoader(path)
    documents.extend(loader.load())

print(f"Loaded {len(documents)} pages from {len(doc_paths)} PDF files")

Loaded 5 pages from 3 PDF files


In [3]:
text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
chunks = text_splitter.split_documents(documents)
print(f"Split documents into {len(chunks)} chunks")

Split documents into 31 chunks


In [5]:
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
persist_directory = "../data/vector_store"
vectordb = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    persist_directory=persist_directory
)

print(f"Vector database presisted to {persist_directory}")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Vector database presisted to ../data/vector_store


In [6]:
model = YOLO('yolov8n.pt')
image_paths = glob.glob("../data/images/*.jpeg")

print("YOLO Inference Results:")
for img_path in image_paths:
    results = model(img_path, verbose=False)
    detected_objects = [model.names[int(box.cls)] for box in results[0].boxes]
    print(f"{os.path.basename(img_path)}: Detected {detected_objects}")

YOLO Inference Results:
busy-construction-site-multiple-workers-safety-vests-hard-hats-actively-engaged-building-activities-scene-410010627.jpeg: Detected ['person', 'person', 'person', 'person', 'person', 'person', 'person', 'chair', 'bench']
istockphoto-1394006687-170667a.jpeg: Detected ['person', 'person']
OIP.jpeg: Detected ['person', 'person', 'person', 'tie']
s-l400.jpeg: Detected ['person']


In [10]:
retriever = vectordb.as_retriever(search_kwargs={"k": 3})
test_questions = [
    "Who is required to pay for personal protective equipment?",
    "What kind of protection is required for head injuries?",
    "How can workers prevent struck-by accidents?",
    "When should workers wear respiratory protection?"
]
print("----- RAG EVALUATION -----")
for q in test_questions:
    retrieved_docs = retriever.invoke(q)
    context = "\n".join([doc.page_content for doc in retrieved_docs])
    prompt = f"""You are a professional workspace saftey assistant. Answer the questions using ONLY the provided context.
                 If the context does not contain the answer, say "I don't know."
                 
                 Context:
                 {context}

                 Question: {q}
                 """
    response = ollama.chat(model='llama3', messages=[{'role': 'user', 'content': prompt}])
    print(f"Q: {q}")
    print(f"A: {response['message']['content']}")
    print(f"Citation: {retrieved_docs[0].metadata['source']} (Page {retrieved_docs[0].metadata['page']})\n")

----- RAG EVALUATION -----
Q: Who is required to pay for personal protective equipment?
A: According to the provided context, employers are required to pay for personal protective equipment.
Citation: ../data/docs\Handout_2_Employers_Must_Provide_and_Pay_for_PPE.pdf (Page 1)

Q: What kind of protection is required for head injuries?
A: According to the provided context, 1910.135: Head protection is the OSHA standard that relates to head injuries.
Citation: ../data/docs\Handout_2_Employers_Must_Provide_and_Pay_for_PPE.pdf (Page 1)

Q: How can workers prevent struck-by accidents?
A: According to the context, workers can prevent struck-by accidents by:

• Never positioning themselves between moving and fixed objects.
Citation: ../data/docs\CONSTRUCTION_HAZARDS_QC.pdf (Page 0)

Q: When should workers wear respiratory protection?
A: I don't know. The context does not contain information about when workers should wear respiratory protection.
Citation: ../data/docs\Handout_2_Employers_Must_Pr